In [1]:
import sys
print(sys.executable)

/Users/farabi/Farabi/ML-Projects/LLM/SST2-Sentiment/.venv/bin/python


In [2]:
import numpy
import pandas
import torch
import transformers

print("Everything is working!")

/Users/farabi/Farabi/ML-Projects/LLM/SST2-Sentiment/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Everything is working!


In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "sst2"

DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

dataset.save_to_disk(str(DATA_PATH))

print("Dataset saved to:", DATA_PATH)

Saving the dataset (1/1 shards): 100%|██████████| 1821/1821 [00:00<00:00, 1370259.70 examples/s]

Dataset saved to: /Users/farabi/Farabi/ML-Projects/LLM/SST2-Sentiment/data/raw/sst2


In [5]:
from datasets import load_from_disk

dataset = load_from_disk(str(DATA_PATH))

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


In [7]:
print(dataset["train"][:5])

{'idx': [0, 1, 2, 3, 4], 'sentence': ['hide new secretions from the parental units ', 'contains no wit , only labored gags ', 'that loves its characters and communicates something rather beautiful about human nature ', 'remains utterly satisfied to remain the same throughout ', 'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up '], 'label': [0, 0, 1, 0, 0]}


In [8]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

In [9]:
# Number of examples belonging to each class
class_counts = train_df["label"].value_counts().sort_index()

print("Class counts:")
print(class_counts)

# Percentage of examples belonging to each class
class_percentages = (
    train_df["label"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("\nClass percentages:")
print(class_percentages)

print("\nTotal training examples:", len(train_df))

Class counts:
label
0    29780
1    37569
Name: count, dtype: int64

Class percentages:
label
0    44.217435
1    55.782565
Name: proportion, dtype: float64

Total training examples: 67349


In [10]:
# Calculate the number of words in each sentence
train_df["word_count"] = train_df["sentence"].str.split().str.len()

# Display descriptive statistics
print(
    train_df["word_count"].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

count    67349.000000
mean         9.409553
std          8.073806
min          1.000000
50%          7.000000
90%         21.000000
95%         26.000000
99%         35.000000
max         52.000000
Name: word_count, dtype: float64


In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [12]:
# Get all training sentences
sentences = train_df["sentence"].tolist()

# Tokenize without truncation or padding
encoded = tokenizer(
    sentences,
    padding=False,
    truncation=False,
    add_special_tokens=True,
    return_length=True
)

# Store token lengths in our DataFrame
train_df["token_count"] = encoded["length"]

In [13]:
print(
    train_df["token_count"].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

count    67349.000000
mean        13.319262
std          9.545841
min          3.000000
50%         10.000000
90%         27.000000
95%         33.000000
99%         44.000000
max         66.000000
Name: token_count, dtype: float64


In [55]:
#We do this to determine what max token_length we should use
#It can be 32 48 64 128 we just use one and check how much it covers

for max_len in [32, 48, 64, 128]:

    truncated = (train_df["token_count"] > max_len).sum()

    percentage = truncated / len(train_df) * 100

    print(
        f"Max length: {max_len} | "
        f"Truncated sentences: {truncated} | "
        f"Percentage: {percentage:.2f}%"
    )

Max length: 32 | Truncated sentences: 3703 | Percentage: 5.50%
Max length: 48 | Truncated sentences: 334 | Percentage: 0.50%
Max length: 64 | Truncated sentences: 4 | Percentage: 0.01%
Max length: 128 | Truncated sentences: 0 | Percentage: 0.00%


In [ ]:
#Here we use dynamic padding, instead of a single tokenizer we use dynamic padding to make 

from transformers import DataCollatorWithPadding

# Select four sentences from SST-2
sentences = train_df["sentence"].iloc[:4].tolist()

# Tokenize each sentence
encoded_sentences = [
    tokenizer(
        sentence,
        truncation=True,
        max_length=128,
        padding=False
    )
    for sentence in sentences
]

# Print original token lengths
for i, encoded in enumerate(encoded_sentences):
    print(f"Sentence {i+1}: {len(encoded['input_ids'])} tokens")

Sentence 1: 10 tokens
Sentence 2: 11 tokens
Sentence 3: 15 tokens
Sentence 4: 10 tokens


In [16]:
sentence = "The movie was absolutely fantastic."

encoded = tokenizer(
    sentence,
    truncation=True,
    max_length=128,
    padding=False
)

print(encoded)

{'input_ids': [101, 1996, 3185, 2001, 7078, 10392, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [17]:
from transformers import DataCollatorWithPadding

# Select four sentences from SST-2
sentences = train_df["sentence"].iloc[:4].tolist()

# Tokenize each sentence
encoded_sentences = [
    tokenizer(
        sentence,
        truncation=True,
        max_length=128,
        padding=False
    )
    for sentence in sentences
]

# Print original token lengths
for i, encoded in enumerate(encoded_sentences):
    print(f"Sentence {i+1}: {len(encoded['input_ids'])} tokens")

Sentence 1: 10 tokens
Sentence 2: 11 tokens
Sentence 3: 15 tokens
Sentence 4: 10 tokens


In [18]:
collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

batch = collator(encoded_sentences)

In [19]:
# Count occurrences of each sentence
sentence_counts = train_df["sentence"].value_counts()

# Number of distinct sentences appearing more than once
duplicate_groups = (sentence_counts > 1).sum()

# Total number of rows belonging to duplicate groups
duplicate_rows = sentence_counts[sentence_counts > 1].sum()

# Number of redundant copies beyond the first occurrence
extra_duplicates = train_df["sentence"].duplicated().sum()

print("Total training rows:", len(train_df))
print("Unique sentences:", train_df["sentence"].nunique())
print("Duplicate groups:", duplicate_groups)
print("Rows in duplicate groups:", duplicate_rows)
print("Extra duplicate rows:", extra_duplicates)

Total training rows: 67349
Unique sentences: 66978
Duplicate groups: 371
Rows in duplicate groups: 742
Extra duplicate rows: 371


In [20]:
# Count distinct labels for each sentence
label_counts = train_df.groupby("sentence")["label"].nunique()

# Identify sentences having more than one distinct label
conflicting_sentences = label_counts[label_counts > 1]

print("Sentences with conflicting labels:", len(conflicting_sentences))

Sentences with conflicting labels: 5


In [21]:
conflicting_examples = train_df[
    train_df["sentence"].isin(conflicting_sentences.index)
]

print(
    conflicting_examples[["sentence", "label"]]
    .sort_values("sentence")
    .head(10)
    .to_string(index=False)
)

        sentence  label
 excruciatingly       0
 excruciatingly       1
      laughably       0
      laughably       1
        sillier       0
        sillier       1
        sleeper       0
        sleeper       1
the only reason       1
the only reason       0


In [22]:
# Total number of sentences with conflicting labels
num_conflicts = len(conflicting_sentences)

# Percentage of duplicate groups containing conflicting labels
conflict_percentage = (
    num_conflicts / duplicate_groups
) * 100

print("Conflicting sentence groups:", num_conflicts)
print(f"Percentage of duplicate groups with conflicts: {conflict_percentage:.2f}%")

Conflicting sentence groups: 5
Percentage of duplicate groups with conflicts: 1.35%


In [23]:
# Load official validation split
validation_df = dataset["validation"].to_pandas()

# Find validation sentences also present in training
overlap = validation_df["sentence"].isin(
    train_df["sentence"]
)

print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))
print("Overlapping validation examples:", overlap.sum())

print(
    "Overlap percentage:",
    f"{overlap.mean() * 100:.2f}%"
)

Training examples: 67349
Validation examples: 872
Overlapping validation examples: 0
Overlap percentage: 0.00%


In [24]:
import pandas as pd

sentences = pd.Series([
    "Amazing Movie",
    "Terrible Movie",
    "amazing movie",
    "Great Acting",
    "TERRIBLE MOVIE"
])

groups = sentences.str.lower().str.strip()

print(groups)

0     amazing movie
1    terrible movie
2     amazing movie
3      great acting
4    terrible movie
dtype: str


In [25]:
from sklearn.model_selection import StratifiedGroupKFold

# Normalize sentences for grouping
groups = (
    train_df["sentence"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Create a group-aware, approximately stratified split
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Select one fold as internal validation
train_idx, val_idx = next(
    sgkf.split(
        X=train_df,
        y=train_df["label"],
        groups=groups
    )
)

internal_train_df = train_df.iloc[train_idx].copy()
internal_val_df = train_df.iloc[val_idx].copy()

print("Internal training examples:", len(internal_train_df))
print("Internal validation examples:", len(internal_val_df))

print("\nTraining class distribution:")
print(internal_train_df["label"].value_counts(normalize=True))

print("\nValidation class distribution:")
print(internal_val_df["label"].value_counts(normalize=True))

# Check group overlap between internal splits
train_groups = set(groups.iloc[train_idx])
val_groups = set(groups.iloc[val_idx])

print("\nOverlapping normalized sentences:",
      len(train_groups.intersection(val_groups)))

Internal training examples: 53879
Internal validation examples: 13470

Training class distribution:
label
1    0.557824
0    0.442176
Name: proportion, dtype: float64

Validation class distribution:
label
1    0.557832
0    0.442168
Name: proportion, dtype: float64

Overlapping normalized sentences: 0


In [26]:
'''
Stage 2: Our goal is to now establish a baseline, so we can compare it against complex models
If our model actually improved 

so first we train a baseline
and then check and increase our model complexity
'''

'\nStage 2: Our goal is to now establish a baseline, so we can compare it against complex models\nIf our model actually improved \n\nso first we train a baseline\nand then check and increase our model complexity\n'

In [27]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

X_train = internal_train_df["sentence"]
y_train = internal_train_df["label"]

# Prepare validation data
X_val = internal_val_df["sentence"]
y_val = internal_val_df["label"]

# Initialize majority-class baseline
baseline = DummyClassifier(strategy="most_frequent")

# Fit on internal training data
baseline.fit(X_train, y_train)

# Predict on internal validation data
y_pred = baseline.predict(X_val)

# Evaluate
accuracy = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average="macro")

print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))

print("\nMajority class:", baseline.classes_[baseline.class_prior_.argmax()])

print(f"\nValidation Accuracy: {accuracy:.4f}")
print(f"Validation Macro F1: {macro_f1:.4f}")

Training examples: 53879
Validation examples: 13470

Majority class: 1

Validation Accuracy: 0.5578
Validation Macro F1: 0.3581


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer()

# Learn vocabulary and IDF values from training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform validation data using the same vocabulary
X_val_tfidf = tfidf.transform(X_val)

print("Training matrix shape:", X_train_tfidf.shape)
print("Validation matrix shape:", X_val_tfidf.shape)

print("Vocabulary size:", len(tfidf.vocabulary_))

Training matrix shape: (53879, 13639)
Validation matrix shape: (13470, 13639)
Vocabulary size: 13639


In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Initialize Logistic Regression
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the model
model.fit(X_train_tfidf, y_train)

# Predict validation labels
y_pred = model.predict(X_val_tfidf)

# Evaluate performance
accuracy = accuracy_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average="macro")

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Macro F1: {macro_f1:.4f}")

Validation Accuracy: 0.8880
Validation Macro F1: 0.8861


In [30]:
from sklearn.metrics import confusion_matrix, classification_report

# Calculate confusion matrix
cm = confusion_matrix(y_val, y_pred, labels=[0, 1])

print("Confusion Matrix:")
print(cm)

# Detailed classification metrics
print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_pred,
        target_names=["Negative", "Positive"]
    )
)

Confusion Matrix:
[[5109  847]
 [ 662 6852]]

Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.86      0.87      5956
    Positive       0.89      0.91      0.90      7514

    accuracy                           0.89     13470
   macro avg       0.89      0.88      0.89     13470
weighted avg       0.89      0.89      0.89     13470



In [31]:
# Create a DataFrame containing validation predictions
error_analysis_df = internal_val_df[
    ["sentence", "label"]
].reset_index(drop=True).copy()

# Add model predictions
error_analysis_df["predicted"] = y_pred

# Keep only incorrectly classified examples
errors_df = error_analysis_df[
    error_analysis_df["label"] != error_analysis_df["predicted"]
]

print("Total misclassified sentences:", len(errors_df))

# Display the first 10 mistakes
print(
    errors_df.head(10).to_string(index=False)
)

Total misclassified sentences: 1509
                                                                                                                                                            sentence  label  predicted
it does n't follow the stale , standard , connect-the-dots storyline which has become commonplace in movies that explore the seamy underbelly of the criminal world       1          0
                                                                                                                                             derivative and hammily       0          1
                                                                                                                            elvira fans could hardly ask for more .       1          0
                                                                                                    the rock 's fighting skills are more in line with steven seagal       1          0
                                                 

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Step 1: Create unigram + bigram features
tfidf_bigram = TfidfVectorizer(
    ngram_range=(1, 2)
)

# Step 2: Fit on training data only
X_train_bigram = tfidf_bigram.fit_transform(X_train)

# Step 3: Transform validation data
X_val_bigram = tfidf_bigram.transform(X_val)

print("Training matrix:", X_train_bigram.shape)
print("Validation matrix:", X_val_bigram.shape)

# Step 4: Train Logistic Regression
model_bigram = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_bigram.fit(X_train_bigram, y_train)

# Step 5: Predict
y_pred_bigram = model_bigram.predict(X_val_bigram)

# Step 6: Evaluate
accuracy_bigram = accuracy_score(y_val, y_pred_bigram)

macro_f1_bigram = f1_score(
    y_val,
    y_pred_bigram,
    average="macro"
)

print(f"\nBigram Validation Accuracy: {accuracy_bigram:.4f}")
print(f"Bigram Validation Macro F1: {macro_f1_bigram:.4f}")

Training matrix: (53879, 78812)
Validation matrix: (13470, 78812)

Bigram Validation Accuracy: 0.9071
Bigram Validation Macro F1: 0.9054


In [33]:
import numpy as np

# Actual validation labels
actual = y_val.to_numpy()

# Check which predictions were correct
unigram_correct = (y_pred == actual)
bigram_correct = (y_pred_bigram == actual)

# Examples fixed by the bigram model
fixed = (~unigram_correct) & bigram_correct

# Examples the bigram model got wrong
# but the unigram model got right
broken = unigram_correct & (~bigram_correct)

print("Mistakes fixed by bigrams:", fixed.sum())
print("New mistakes introduced:", broken.sum())

print(
    "Net improvement in correct predictions:",
    fixed.sum() - broken.sum()
)

Mistakes fixed by bigrams: 425
New mistakes introduced: 168
Net improvement in correct predictions: 257


In [34]:
# Create a DataFrame for comparing predictions
comparison_df = internal_val_df[
    ["sentence", "label"]
].reset_index(drop=True).copy()

# Add predictions from both models
comparison_df["unigram_pred"] = y_pred
comparison_df["bigram_pred"] = y_pred_bigram

# Extract sentences corrected by bigrams
fixed_examples = comparison_df.loc[fixed]

print("Total corrected examples:", len(fixed_examples))

print(
    fixed_examples.head(10).to_string(index=False)
)

Total corrected examples: 425
                                                                                                                                                            sentence  label  unigram_pred  bigram_pred
it does n't follow the stale , standard , connect-the-dots storyline which has become commonplace in movies that explore the seamy underbelly of the criminal world       1             0            1
                                                                                                                            elvira fans could hardly ask for more .       1             0            1
                                                                                                                           handsome but unfulfilling suspense drama       0             1            0
                                                                                                                                                    very funny joke       1   

In [35]:
# Select the first corrected sentence
sentence = fixed_examples.iloc[0]["sentence"]

print("Sentence:", sentence)

# Convert the sentence to TF-IDF features
vector = tfidf_bigram.transform([sentence])

# Get vocabulary feature names
feature_names = tfidf_bigram.get_feature_names_out()

# Get learned Logistic Regression weights
weights = model_bigram.coef_[0]

# Calculate each feature's contribution
contributions = vector.multiply(weights).tocoo()

# Combine feature names with their contributions
feature_contributions = [
    (feature_names[col], value)
    for col, value in zip(contributions.col, contributions.data)
]

# Sort by absolute contribution
feature_contributions.sort(
    key=lambda x: abs(x[1]),
    reverse=True
)

# Display the most influential features
for feature, contribution in feature_contributions[:15]:
    print(f"{feature:35s} {contribution:+.4f}")

Sentence: it does n't follow the stale , standard , connect-the-dots storyline which has become commonplace in movies that explore the seamy underbelly of the criminal world 
stale                               -0.3589
world                               +0.3031
does                                -0.2596
underbelly                          -0.1572
movies that                         +0.1316
which                               -0.1197
explore                             +0.1190
it does                             +0.0948
underbelly of                       -0.0769
the stale                           -0.0753
movies                              +0.0716
storyline                           -0.0659
storyline which                     +0.0642
does follow                         +0.0597
the                                 +0.0561


In [36]:
# Sum all feature contributions
total_contribution = contributions.data.sum()

# Get Logistic Regression's bias
bias = model_bigram.intercept_[0]

# Calculate final decision score
z = total_contribution + bias

# Get probability of positive sentiment
probability = model_bigram.predict_proba(vector)[0, 1]

print("Total feature contribution:", total_contribution)
print("Bias:", bias)
print("Final decision score:", z)
print("Positive probability:", probability)

print("Predicted class:", model_bigram.predict(vector)[0])

Total feature contribution: 0.005383489188675372
Bias: 0.23158190301742487
Final decision score: 0.23696539220610024
Positive probability: 0.5589656829170196
Predicted class: 1


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

sentence = "it does n't follow the stale storyline"

# Original TF-IDF
tfidf_original = TfidfVectorizer(
    ngram_range=(1, 2)
)

# Modified TF-IDF
tfidf_modified = TfidfVectorizer(
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+(?:['’]\w+)?\b"
)

print("Original tokenizer:")
print(tfidf_original.build_analyzer()(sentence))

print("\nModified tokenizer:")
print(tfidf_modified.build_analyzer()(sentence))

Original tokenizer:
['it', 'does', 'follow', 'the', 'stale', 'storyline', 'it does', 'does follow', 'follow the', 'the stale', 'stale storyline']

Modified tokenizer:
['it', 'does', "n't", 'follow', 'the', 'stale', 'storyline', 'it does', "does n't", "n't follow", 'follow the', 'the stale', 'stale storyline']


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Modified TF-IDF
tfidf_modified = TfidfVectorizer(
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+(?:['’]\w+)?\b"
)

# Fit only on training data
X_train_modified = tfidf_modified.fit_transform(X_train)

# Transform validation data
X_val_modified = tfidf_modified.transform(X_val)

print("Training matrix:", X_train_modified.shape)
print("Validation matrix:", X_val_modified.shape)

# Train Logistic Regression
model_modified = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_modified.fit(X_train_modified, y_train)

# Predict
y_pred_modified = model_modified.predict(X_val_modified)

# Evaluate
accuracy_modified = accuracy_score(y_val, y_pred_modified)

macro_f1_modified = f1_score(
    y_val,
    y_pred_modified,
    average="macro"
)

print(f"\nModified TF-IDF Accuracy: {accuracy_modified:.4f}")
print(f"Modified TF-IDF Macro F1: {macro_f1_modified:.4f}")

Training matrix: (53879, 79181)
Validation matrix: (13470, 79181)

Modified TF-IDF Accuracy: 0.9099
Modified TF-IDF Macro F1: 0.9083


In [39]:
import numpy as np

actual = y_val.to_numpy()

# Check predictions from both models
bigram_correct = (y_pred_bigram == actual)

modified_correct = (y_pred_modified == actual)

# Mistakes fixed by modified tokenizer
fixed = (~bigram_correct) & modified_correct

# New mistakes introduced by modified tokenizer
broken = bigram_correct & (~modified_correct)

print("Mistakes fixed:", fixed.sum())
print("New mistakes:", broken.sum())
print("Net improvement:", fixed.sum() - broken.sum())

Mistakes fixed: 104
New mistakes: 65
Net improvement: 39


In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

C_values = [0.1, 1.0, 10.0]

for C in C_values:

    model = LogisticRegression(
        C=C,
        max_iter=1000,
        random_state=42
    )

    # Train
    model.fit(X_train_modified, y_train)

    # Predict training and validation data
    train_pred = model.predict(X_train_modified)
    val_pred = model.predict(X_val_modified)

    # Evaluate
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    val_f1 = f1_score(y_val, val_pred, average="macro")

    print(f"\nC = {C}")
    print(f"Training Accuracy:   {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(f"Validation Macro F1: {val_f1:.4f}")


C = 0.1
Training Accuracy:   0.8376
Validation Accuracy: 0.8120
Validation Macro F1: 0.8001

C = 1.0
Training Accuracy:   0.9538
Validation Accuracy: 0.9099
Validation Macro F1: 0.9083

C = 10.0
Training Accuracy:   0.9857
Validation Accuracy: 0.9281
Validation Macro F1: 0.9271


In [41]:
model_c100 = LogisticRegression(
    C=100.0,
    max_iter=1000,
    random_state=42
)

model_c100.fit(X_train_modified, y_train)

train_pred = model_c100.predict(X_train_modified)
val_pred = model_c100.predict(X_val_modified)

print(
    "Training Accuracy:",
    accuracy_score(y_train, train_pred)
)

print(
    "Validation Accuracy:",
    accuracy_score(y_val, val_pred)
)

print(
    "Validation Macro F1:",
    f1_score(y_val, val_pred, average="macro")
)

Training Accuracy: 0.9917036322129216
Validation Accuracy: 0.9264291017074981
Validation Macro F1: 0.92537487862028


In [42]:
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Create the same group-aware splitter
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_accuracies = []
cv_f1_scores = []

# Iterate through all 5 folds
for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(
        X=train_df["sentence"],
        y=train_df["label"],
        groups=groups
    ),
    start=1
):

    # Get training and validation data for this fold
    X_train_fold = train_df["sentence"].iloc[train_idx]
    X_val_fold = train_df["sentence"].iloc[val_idx]

    y_train_fold = train_df["label"].iloc[train_idx]
    y_val_fold = train_df["label"].iloc[val_idx]

    # Create a NEW TF-IDF vectorizer for this fold
    tfidf_fold = TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b\w+(?:['’]\w+)?\b"
    )

    # Fit only on this fold's training data
    X_train_vec = tfidf_fold.fit_transform(X_train_fold)
    X_val_vec = tfidf_fold.transform(X_val_fold)

    # Create and train a NEW model
    model_fold = LogisticRegression(
        C=100,
        max_iter=1000,
        random_state=42
    )

    model_fold.fit(X_train_vec, y_train_fold)

    # Make predictions
    predictions = model_fold.predict(X_val_vec)

    # Evaluate
    accuracy = accuracy_score(y_val_fold, predictions)
    macro_f1 = f1_score(
        y_val_fold,
        predictions,
        average="macro"
    )

    cv_accuracies.append(accuracy)
    cv_f1_scores.append(macro_f1)

    print(
        f"Fold {fold}: "
        f"Accuracy = {accuracy:.4f}, "
        f"Macro F1 = {macro_f1:.4f}"
    )

# Calculate overall results
print("\nCross-Validation Results")
print(f"Mean Accuracy: {np.mean(cv_accuracies):.4f}")
print(f"Std Accuracy:  {np.std(cv_accuracies):.4f}")
print(f"Mean Macro F1: {np.mean(cv_f1_scores):.4f}")

Fold 1: Accuracy = 0.9264, Macro F1 = 0.9254
Fold 2: Accuracy = 0.9252, Macro F1 = 0.9243
Fold 3: Accuracy = 0.9232, Macro F1 = 0.9221
Fold 4: Accuracy = 0.9258, Macro F1 = 0.9248
Fold 5: Accuracy = 0.9252, Macro F1 = 0.9242

Cross-Validation Results
Mean Accuracy: 0.9252
Std Accuracy:  0.0011
Mean Macro F1: 0.9242


In [43]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Create the complete model pipeline
final_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b\w+(?:['’]\w+)?\b"
    )),

    ("classifier", LogisticRegression(
        C=10,
        max_iter=1000,
        random_state=42
    ))
])

# Train on ALL 67,349 training examples
final_model.fit(
    train_df["sentence"],
    train_df["label"]
)

# Load official validation data
official_val_df = dataset["validation"].to_pandas()

# Predict official validation labels
official_predictions = final_model.predict(
    official_val_df["sentence"]
)

# Evaluate
official_accuracy = accuracy_score(
    official_val_df["label"],
    official_predictions
)

official_f1 = f1_score(
    official_val_df["label"],
    official_predictions,
    average="macro"
)

print(f"Official Validation Accuracy: {official_accuracy:.4f}")
print(f"Official Validation Macro F1: {official_f1:.4f}")

Official Validation Accuracy: 0.8177
Official Validation Macro F1: 0.8168


In [44]:
# Calculate word lengths
train_lengths = train_df["sentence"].str.split().str.len()

official_val_lengths = (
    official_val_df["sentence"].str.split().str.len()
)

# Compare distributions
import pandas as pd

length_comparison = pd.DataFrame({
    "Training": train_lengths.describe(
        percentiles=[0.50, 0.90, 0.95]
    ),
    "Official Validation": official_val_lengths.describe(
        percentiles=[0.50, 0.90, 0.95]
    )
})

print(
    length_comparison.loc[
        ["count", "mean", "50%", "90%", "95%", "max"]
    ]
)

           Training  Official Validation
count  67349.000000           872.000000
mean       9.409553            19.548165
50%        7.000000            19.000000
90%       21.000000            31.900000
95%       26.000000            34.450000
max       52.000000            47.000000


In [45]:
import pandas as pd

# Create a DataFrame for error analysis
analysis_df = official_val_df[
    ["sentence", "label"]
].copy()

# Add model predictions
analysis_df["predicted"] = official_predictions

# Calculate sentence lengths
analysis_df["word_count"] = (
    analysis_df["sentence"].str.split().str.len()
)

# Group sentences based on length
analysis_df["length_group"] = pd.cut(
    analysis_df["word_count"],
    bins=[0, 10, 20, 30, float("inf")],
    labels=["1-10", "11-20", "21-30", "31+"]
)

# Identify correct predictions
analysis_df["correct"] = (
    analysis_df["label"] == analysis_df["predicted"]
)

# Calculate accuracy for each length group
results = analysis_df.groupby(
    "length_group",
    observed=True
).agg(
    examples=("correct", "size"),
    accuracy=("correct", "mean")
)

print(results)

              examples  accuracy
length_group                    
1-10               152  0.881579
11-20              340  0.823529
21-30              275  0.796364
31+                105  0.761905


In [46]:
results_by_class = analysis_df.groupby(
    ["length_group", "label"],
    observed=True
).agg(
    examples=("correct", "size"),
    accuracy=("correct", "mean")
)

print(results_by_class)

                    examples  accuracy
length_group label                    
1-10         0            83  0.879518
             1            69  0.884058
11-20        0           165  0.751515
             1           175  0.891429
21-30        0           126  0.730159
             1           149  0.852349
31+          0            54  0.685185
             1            51  0.843137


In [47]:
# Select long negative sentences predicted as positive
long_negative_errors = analysis_df[
    (analysis_df["word_count"] > 20) &
    (analysis_df["label"] == 0) &
    (analysis_df["predicted"] == 1)
]

print("Total long negative errors:", len(long_negative_errors))

print(
    long_negative_errors[
        ["sentence", "word_count", "label", "predicted"]
    ].head(10).to_string(index=False)
)

Total long negative errors: 51
                                                                                                                                                                                                   sentence  word_count  label  predicted
                              pumpkin takes an admirable look at the hypocrisy of political correctness , but it does so with such an uneven tone that you never know when humor ends and tragedy begins .           32      0          1
                                                             ... designed to provide a mix of smiles and tears , `` crossroads '' instead provokes a handful of unintentional howlers and numerous yawns .           25      0          1
                                                   ( w ) hile long on amiable monkeys and worthy environmentalism , jane goodall 's wild chimpanzees is short on the thrills the oversize medium demands .           27      0          1
                                 

In [48]:
for name, df in [
    ("Training", train_df),
    ("Official Validation", official_val_df)
]:

    # Calculate sentence lengths
    lengths = df["sentence"].str.split().str.len()

    # Identify sentences with more than 20 words
    is_long = lengths > 20

    # Group by sentiment label
    results = (
        df.assign(is_long=is_long)
        .groupby("label")["is_long"]
        .agg(["sum", "count", "mean"])
    )

    print(f"\n{name}")
    print(results)


Training
        sum  count      mean
label                       
0      3526  29780  0.118402
1      3846  37569  0.102372

Official Validation
       sum  count      mean
label                      
0      180    428  0.420561
1      200    444  0.450450


In [49]:
print(X_val_modified[1])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 45 stored elements and shape (1, 79181)>
  Coords	Values
  (0, 391)	0.0442249964052367
  (0, 1828)	0.1644724869353001
  (0, 4520)	0.07350138311669865
  (0, 4698)	0.17921060799223693
  (0, 8136)	0.0745466442394893
  (0, 8403)	0.1975649237609815
  (0, 11462)	0.17921060799223693
  (0, 11463)	0.20185126404910067
  (0, 13497)	0.10111236377621059
  (0, 13587)	0.1625173541936879
  (0, 19052)	0.17093574565449224
  (0, 19055)	0.19110166504178933
  (0, 19827)	0.10685930946470684
  (0, 19905)	0.16517106906987453
  (0, 22084)	0.1218142434374765
  (0, 22124)	0.1777870759885447
  (0, 25509)	0.07314946375516707
  (0, 25744)	0.1371692434705047
  (0, 28187)	0.15911871140019837
  (0, 28190)	0.19406272874044017
  (0, 32135)	0.11679743902970692
  (0, 32140)	0.20185126404910067
  (0, 46703)	0.04768527919510716
  (0, 47867)	0.17093574565449224
  (0, 49506)	0.09280711562944284
  (0, 49507)	0.16379842157699395
  (0, 50607)	0.19110166504178933
  (0,

In [50]:
from sklearn.linear_model import LogisticRegression

# Train C=10 on our original internal split
model_c10 = LogisticRegression(
    C=10,
    max_iter=1000,
    random_state=42
)

model_c10.fit(X_train_modified, y_train)

# Predict internal validation examples
internal_predictions = model_c10.predict(X_val_modified)

In [51]:
# Create internal validation analysis
internal_analysis = internal_val_df[
    ["sentence", "label"]
].copy()

internal_analysis["predicted"] = internal_predictions

internal_analysis["word_count"] = (
    internal_analysis["sentence"].str.split().str.len()
)

# Create sentence-length groups
internal_analysis["length_group"] = pd.cut(
    internal_analysis["word_count"],
    bins=[0, 10, 20, 30, float("inf")],
    labels=["1-10", "11-20", "21-30", "31+"]
)

# Identify correct predictions
internal_analysis["correct"] = (
    internal_analysis["label"] ==
    internal_analysis["predicted"]
)

# Calculate accuracy for each group
internal_results = internal_analysis.groupby(
    "length_group",
    observed=True
).agg(
    examples=("correct", "size"),
    accuracy=("correct", "mean")
)

print(internal_results)

              examples  accuracy
length_group                    
1-10              9029  0.919371
11-20             3006  0.944112
21-30             1126  0.952931
31+                309  0.938511


In [52]:
internal_by_class = internal_analysis.groupby(
    ["length_group", "label"],
    observed=True
).agg(
    examples=("correct", "size"),
    accuracy=("correct", "mean")
)

print(internal_by_class)

                    examples  accuracy
length_group label                    
1-10         0          3833  0.900861
             1          5196  0.933025
11-20        0          1441  0.941013
             1          1565  0.946965
21-30        0           562  0.944840
             1           564  0.960993
31+          0           120  0.925000
             1           189  0.947090


In [53]:
def normalize_text(text):
    return " ".join(text.lower().split())


def count_substring_matches(validation_df, training_df):

    # Select long negative sentences
    samples = validation_df[
        (validation_df["label"] == 0) &
        (validation_df["sentence"].str.split().str.len() > 20)
    ].sample(n=50, random_state=42)

    # Normalize training sentences
    training_texts = [
        " " + normalize_text(s) + " "
        for s in training_df["sentence"]
    ]

    matches = 0

    for sentence in samples["sentence"]:

        target = " " + normalize_text(sentence) + " "

        # Check whether the entire sentence appears
        # inside any training example
        if any(target in text for text in training_texts):
            matches += 1

    return matches


internal_matches = count_substring_matches(
    internal_val_df,
    internal_train_df
)

official_matches = count_substring_matches(
    official_val_df,
    train_df
)

print("Internal validation matches:", internal_matches, "/ 50")
print("Official validation matches:", official_matches, "/ 50")

Internal validation matches: 25 / 50
Official validation matches: 0 / 50


In [54]:
# Select the same 50 long negative validation examples
sample = internal_analysis[
    (internal_analysis["label"] == 0) &
    (internal_analysis["word_count"] > 20)
].sample(n=50, random_state=42).copy()

# Normalize training sentences
training_texts = [
    " " + normalize_text(s) + " "
    for s in internal_train_df["sentence"]
]

# Check whether each validation sentence appears
# inside a training sentence
def is_contained(sentence):
    target = " " + normalize_text(sentence) + " "

    return any(target in text for text in training_texts)


sample["matched"] = sample["sentence"].apply(is_contained)

# Compare prediction accuracy
results = sample.groupby("matched").agg(
    examples=("correct", "size"),
    accuracy=("correct", "mean")
)

print(results)

         examples  accuracy
matched                    
False          25      0.96
True           25      0.96


###Transformer Part

In [58]:
from datasets import Dataset

train_bert_ds = Dataset.from_pandas(
    internal_train_df[["sentence", "label"]],
    preserve_index=False
)

val_bert_ds = Dataset.from_pandas(
    internal_val_df[["sentence", "label"]],
    preserve_index=False
)

print(train_bert_ds)
print(val_bert_ds)

Dataset({
    features: ['sentence', 'label'],
    num_rows: 53879
})
Dataset({
    features: ['sentence', 'label'],
    num_rows: 13470
})


In [59]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [60]:
def tokenize_function(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=128,
        padding=False
    )

In [61]:
train_bert_ds = train_bert_ds.map(
    tokenize_function,
    batched=True
)

val_bert_ds = val_bert_ds.map(
    tokenize_function,
    batched=True
)

Map: 100%|██████████| 13470/13470 [00:00<00:00, 143153.21 examples/s]


In [62]:
train_bert_ds = train_bert_ds.map(
    tokenize_function,
    batched=True
)

val_bert_ds = val_bert_ds.map(
    tokenize_function,
    batched=True
)

Map: 100%|██████████| 13470/13470 [00:00<00:00, 80799.04 examples/s]


In [63]:
print(train_bert_ds[0])

{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [64]:
train_bert_ds = train_bert_ds.remove_columns(["sentence"])
val_bert_ds = val_bert_ds.remove_columns(["sentence"])

In [65]:
train_bert_ds = train_bert_ds.rename_column("label", "labels")
val_bert_ds = val_bert_ds.rename_column("label", "labels")

In [66]:
print(train_bert_ds.column_names)

['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [68]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_bert_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator
)

val_loader = DataLoader(
    val_bert_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator
)

In [71]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)

print("Padded batch shape:")
print(batch["input_ids"].shape)

print("\nActual token lengths:")
print(batch["attention_mask"].sum(dim=1))

labels torch.Size([16])
input_ids torch.Size([16, 50])
token_type_ids torch.Size([16, 50])
attention_mask torch.Size([16, 50])
Padded batch shape:
torch.Size([16, 50])

Actual token lengths:
tensor([17, 15, 11, 50, 28, 10,  4,  8,  9, 11, 29, 20, 10,  6,  7, 14])


In [72]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "bert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(type(model).__name__)
print("Number of labels:", model.config.num_labels)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21435.23it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

BertForSequenceClassification
Number of labels: 2


In [73]:
import torch

# Disable training-specific behavior
model.eval()

# Select only 2 examples from our existing batch
small_batch = {
    key: value[:2]
    for key, value in batch.items()
}

# Perform a forward pass without calculating gradients
with torch.no_grad():
    outputs = model(**small_batch)

print("Logits shape:", outputs.logits.shape)
print("Logits:", outputs.logits)
print("Loss:", outputs.loss.item())

Logits shape: torch.Size([2, 2])
Logits: tensor([[ 0.0409, -0.2869],
        [ 0.0137, -0.4227]])
Loss: 0.7387818694114685


In [74]:
import torch

# Convert logits into probabilities
probabilities = torch.softmax(outputs.logits, dim=-1)

# Select the class with the highest probability
predictions = torch.argmax(probabilities, dim=-1)

# Actual sentiment labels
actual_labels = small_batch["labels"]

print("Probabilities:")
print(probabilities)

print("\nPredicted classes:")
print(predictions)

print("\nActual classes:")
print(actual_labels)

Probabilities:
tensor([[0.5812, 0.4188],
        [0.6074, 0.3926]])

Predicted classes:
tensor([0, 0])

Actual classes:
tensor([0, 1])
